# Which groups are interchangeable

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/igbaskin/cmf-qsar/blob/main/docs/tutorials/28_interchangeable_groups.ipynb)

A medicinal chemist who asks for a bioisostere table is not asking which group is best. The
question is **which groups can replace each other** at a position, so that one of them can be
chosen for something the model does not describe: solubility, cost, a patent, a metabolic
liability.

The program answers it without fitting a model once per group. The prediction of the overlap
kernel is a sum over the atoms, so the part of it that belongs to a substituent is the sum over
that substituent's own atoms, and exchanging group A for group B is the difference of two such
sums. That costs one integral per pair of groups.

The notebook asks three questions.

1. How are the pairs found, and what does the model say each exchange does?
2. Does the predicted change rank the measured activity difference better than two distances
   between the groups: their fingerprint distance, and the distance between their fields in the
   poses they occupy?
3. The reference that matters most: a **model** that needs no field at all. The two distances are
   not fitted to anything, so what the co-field gains over them could be the fitting rather than
   the field. A kernel ridge model on fingerprints of the whole molecules, refitted the same way,
   separates the two.

The data are the glycogen phosphorylase set of the Sutherland collection, 66 molecules with a
measured pK<sub>i</sub>. Running time: about two minutes.

In [1]:
# Run this cell first. On Google Colab it fetches the repository and installs the package;
# on a local checkout it only moves to the repository root, so that the example paths work.
import os, subprocess, sys

if "google.colab" in sys.modules:
    if not os.path.exists("cmf-qsar"):
        subprocess.run(["git", "clone", "-q", "https://github.com/igbaskin/cmf-qsar.git"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./cmf-qsar", "py3Dmol"], check=True)
    os.chdir("cmf-qsar")
else:
    while not os.path.exists("pyproject.toml") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")
print("working in", os.getcwd())

working in C:\Users\baski\work\claude\cmf


## The pairs

A **matched molecular pair** is two molecules that differ in one group at one position. RDKit's
`rdMMPA` finds them: it cuts one single bond at a time, tries both pieces as the core, and keeps
the pairs that share a core. Nothing is chosen by eye. Two molecules that match on several cores
are counted once.

The cell finds the pairs of this set and prints a few, with the two groups and the measured
difference in activity. The groups are written as SMILES with `[*:1]` for the attachment
point.

In [2]:
import csv
import json
import time

import numpy as np
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
sys.path.insert(0, "benchmarks")
import run_bioisostere as RB                       # the benchmark script of plan item MC1
from cmfqsar.bioisostere import matched_pairs, rank_gain, score_pairs, spearman

start = time.time()
mols, names, y, what = RB.load("GPB")
pairs = matched_pairs(mols, y, names)
print("%d molecules (%s), %d matched pairs" % (len(mols), what, len(pairs)))
print("the activity runs from %.2f to %.2f" % (y.min(), y.max()))
print()
for p in pairs[:6]:
    print("  %-22s -> %-22s measured %+.2f" % (p.groups[0], p.groups[1], p.delta))
sizes = np.abs([p.delta for p in pairs])
print()
print("the measured difference of a pair: median %.2f, and %.0f %% of the pairs are under 0.5"
      % (np.median(sizes), 100 * np.mean(sizes < 0.5)))

66 molecules (glycogen phosphorylase b, 66 molecules), 730 matched pairs
the activity runs from 1.30 to 6.80

  O[*:1]                 -> [NH3+]C[*:1]           measured -1.30
  O[*:1]                 -> C[*:1]                 measured -1.50
  O[*:1]                 -> NC(=O)[*:1]            measured +0.60
  O[*:1]                 -> c1ccc2[nH]c(O[*:1])cc2c1 measured -0.20
  O[*:1]                 -> NNC(=O)[*:1]           measured -0.30
  O[*:1]                 -> O=C(Nc1ccccc1)[*:1]    measured -0.90

the measured difference of a pair: median 0.80, and 30 % of the pairs are under 0.5


66 molecules give 730 pairs, because one molecule belongs to as many pairs as it shares a core
with. The exchanges are real medicinal chemistry: a hydroxyl for an amide, an amide for a longer
one, a small group for a ring.

Half of the pairs differ by less than 0.8 log units in activity, and 30 % by less than 0.5. Those
are the pairs a chemist would call interchangeable. The question of this notebook is which
measure puts them at the bottom of the list.

## Four measures of an exchange

Each measure gets the same pairs and is compared with the **size** of the measured activity
difference, by rank correlation. Two groups that can replace each other differ little in
activity, so a measure that ranks that well is a measure a chemist can use.

- **fingerprint distance**: one minus the Tanimoto similarity of the two groups. No geometry, no
  model.
- **field distance**: the distance between the fields of the two groups in the poses they
  occupy, restricted to their own atoms. It has the geometry but no model, and it already knows
  the shape and the charge each group puts where.
- **the fingerprint model**: kernel ridge on Morgan fingerprints of the whole molecules. It is a
  model, but it knows nothing about where in space the two groups differ. Its predicted change is
  the difference of its two predictions.
- **the co-field**: the field model's predicted change, the difference of two sums over the atoms
  of a group.

Both models are **refitted without both members of every pair**, so neither has seen the two
activities it is judged against.

In [3]:
model, fields, K, q2, alpha, lam = RB.fitted(mols, names, y)
print("the field model: q2 %+.3f at alpha %g, ridge %g" % (q2, alpha, lam))
honest = score_pairs(model, fields, pairs, coef_of=RB.leave_out(K, y, lam))
fp_change, fp_q2, fp_lam = RB.fingerprint_changes(mols, y, pairs)
print("the fingerprint model: q2 %+.3f at ridge %g" % (fp_q2, fp_lam))

measured = np.abs([s.measured for s in honest])
columns = {"fingerprint distance": [s.fingerprint for s in honest],
           "field distance": [s.field for s in honest],
           "the fingerprint model": [abs(c) for c in fp_change],
           "the co-field": [abs(s.change) for s in honest]}
print()
print("%-24s %28s" % ("measure", "rank correlation with the size"))
for name, col in columns.items():
    print("%-24s %27.3f " % (name, spearman(col, measured)))
print()
print("the co-field against each of the others, with a bootstrap interval over the pairs:")
for name in ("fingerprint distance", "field distance", "the fingerprint model"):
    g = rank_gain(columns["the co-field"], columns[name], measured, seed=0)
    print("  %-24s %+.3f [%+.3f, %+.3f]" % (name, g[0], g[1], g[2]))
print("(%.0f s)" % (time.time() - start))

the field model: q2 +0.533 at alpha 2, ridge 1


the fingerprint model: q2 +0.448 at ridge 0.1

measure                  rank correlation with the size
fingerprint distance                          -0.025 
field distance                                 0.005 
the fingerprint model                          0.522 
the co-field                                   0.561 

the co-field against each of the others, with a bootstrap interval over the pairs:


  fingerprint distance     +0.586 [+0.505, +0.669]


  field distance           +0.556 [+0.479, +0.632]


  the fingerprint model    +0.039 [-0.026, +0.103]
(83 s)


**Neither distance ranks anything on this set.** The fingerprint distance reaches -0.025 and the
field distance +0.005. The field distance is not a weak reference: it compares the two groups in
the poses they occupy, so it knows the shape and the charge each of them puts where. It still
says nothing about the activity, and the reason is plain once stated. A distance measures how
different two groups are. Whether that difference *matters* depends on the place they sit in, and
no distance between the groups alone can know it.

**The co-field ranks it well:** +0.561, ahead of both distances by about +0.56, with intervals
far from zero. That is the comparison the plan item made, and on it the co-field clearly earns
its place.

**But a model that needs no field does nearly as well:** +0.522. The difference is +0.039
[-0.026, +0.103], an interval that contains zero. The fingerprint model does not know where the
two groups differ, nor what shape or charge they have in the pocket. It only knows which
molecules are alike and what they measured.

So what the co-field gains over a distance is mostly the **fitting**, not the field. A distance
is not fitted to anything; both models are.

## The three sets

The benchmark runs the same comparison on three sets: glycogen phosphorylase (GPB, 730 pairs),
thermolysin (THER, 142 pairs) and thrombin (THR, 751 pairs).

In [4]:
rows = []
for key in ("GPB", "THER", "THR"):
    with open("benchmarks/results/bioisostere/%s.json" % key, encoding="utf-8") as fh:
        rows.append(json.load(fh))
print("%-5s %6s %12s %10s %12s %10s %12s" % ("set", "pairs", "fingerprint", "field",
                                             "fp model", "co-field", "whole model"))
for d in rows:
    rho = {r["name"]: r["rho"] for r in d["rows"]}
    print("%-5s %6d %12.3f %10.3f %12.3f %10.3f %12.3f"
          % (d["set"], d["pairs"], rho["fingerprint distance"], rho["field distance"],
             rho[RB.MODEL_REFERENCE], rho["predicted change, the pair left out"],
             rho["predicted change, the whole model"]))
print()
print("the co-field minus each reference, with its interval:")
for d in rows:
    print("  %-5s " % d["set"] + "; ".join(
        "%s %+.3f [%+.3f, %+.3f]" % (k.replace("predicted change of a fingerprint model, the pair left out",
                                               "fingerprint model"), *d["gains"][k])
        for k in d["gains"]))
print()
print("the live run repeats the stored one:",
      all(abs(spearman(columns[a], measured) - {r["name"]: r["rho"] for r in rows[0]["rows"]}[b]) < 1e-12
          for a, b in (("fingerprint distance", "fingerprint distance"),
                       ("field distance", "field distance"),
                       ("the fingerprint model", RB.MODEL_REFERENCE),
                       ("the co-field", "predicted change, the pair left out"))))

set    pairs  fingerprint      field     fp model   co-field  whole model
GPB      730       -0.025      0.005        0.522      0.561        0.631
THER     142        0.084      0.162        0.398      0.277        0.641
THR      751        0.144      0.242        0.392      0.335        0.711

the co-field minus each reference, with its interval:
  GPB   fingerprint distance +0.586 [+0.505, +0.669]; field distance +0.556 [+0.479, +0.632]; fingerprint model +0.039 [-0.026, +0.103]
  THER  fingerprint distance +0.193 [-0.059, +0.437]; field distance +0.116 [-0.089, +0.313]; fingerprint model -0.121 [-0.281, +0.030]
  THR   fingerprint distance +0.191 [+0.106, +0.279]; field distance +0.093 [+0.008, +0.181]; fingerprint model -0.057 [-0.132, +0.022]

the live run repeats the stored one: True


The three sets agree.

- **Against the two distances** the co-field is ahead on GPB and THR by more than the interval,
  and ahead on THER with an interval that contains zero. That is the published reading of the
  item.
- **Against the fingerprint model** it is ahead on GPB by +0.039 and behind on THER by -0.121
  and on THR by -0.057. Every one of those three intervals contains zero. On no set does the
  field model clearly beat a model that has no field.
- **The last column is a warning.** The whole model, which has seen both activities of every
  pair it is judging, reaches +0.63 to +0.71. Refitting without the two members of each pair
  leaves +0.28 to +0.56. The gap between the two columns is memory, and it is large. A table
  built without that care would look twice as good as it is.

So the measurement that the item reported stands, and the reference added here narrows what it
shows: a bioisostere table is worth computing **from a model**, and on these sets the model does
not have to be a field model.

## The part of the table that holds

The ranking is one question. The number a chemist reads off the table is the other: how large is
the predicted change, and is it larger than the model's own error?

`build_table` writes, for one core, the predicted change of every pair of groups found on it. A
change smaller than twice the conformal half width is a change this model cannot see, and those
are the pairs the table calls interchangeable. The cell reads the stored table of 35 groups on
the glucose core of this set.

In [5]:
with open("benchmarks/results/bioisostere/table_GPB.csv", encoding="utf-8") as fh:
    table = list(csv.DictReader(fh))
half = rows[0]["half_width"]
unordered = {}
for r in table:
    key = tuple(sorted((r["group a"], r["group b"])))
    unordered.setdefault(key, abs(float(r["predicted change"])))
print("%d groups on the core %s, so %d pairs of groups"
      % (len(rows[0]["table"]["groups"]), rows[0]["table"]["core"], len(unordered)))
print("the conformal half width of this model is %.2f log units at 90 %%, so a predicted change"
      % half)
print("under %.2f is one the model cannot distinguish from no change at all" % (2 * half))
print()
order = sorted(unordered.items(), key=lambda kv: kv[1])
for title, part in (("the pairs whose exchange it predicts the least", order[:4]),
                    ("the pairs whose exchange it predicts the most", order[-4:][::-1])):
    print(title)
    for (a, b), change in part:
        print("  %-24s %-24s predicted change %.2f" % (a, b, change))
seen = sum(1 for v in unordered.values() if v >= 2 * half)
print()
print("%d of the %d pairs of groups are separated by more than twice the half width;"
      % (seen, len(unordered)))
print("the largest predicted change on this core is %.2f" % max(unordered.values()))
groups = rows[0]["table"]["groups"]
same = [g for g, v in rows[0]["table"]["interchangeable"].items() if v]
print("%d of the %d groups have at least one partner the model cannot tell them apart from"
      % (len(same), len(groups)))

35 groups on the core OC[C@H]1O[C@@H]([*:1])[C@H](O)[C@@H](O)[C@@H]1O, so 595 pairs of groups
the conformal half width of this model is 1.30 log units at 90 %, so a predicted change
under 2.60 is one the model cannot distinguish from no change at all

the pairs whose exchange it predicts the least
  CNNC(=O)[*:1]            NNC(=O)[*:1]             predicted change 0.00
  CO[*:1]                  c1ccc2[nH]c(O[*:1])cc2c1 predicted change 0.00
  CC(=O)NCC(=O)N[*:1]      O=C(NC1CC1)[*:1]         predicted change 0.00
  [N-]=[N+]=NC[*:1]        [NH3+]CC[*:1]            predicted change 0.00
the pairs whose exchange it predicts the most
  NC(=O)CS[*:1]            O=C(CCl)N[*:1]           predicted change 2.09
  NC(=O)CS[*:1]            O=C(CBr)N[*:1]           predicted change 2.07
  COC(=O)N[*:1]            NC(=O)CS[*:1]            predicted change 2.06
  CCC(=O)N[*:1]            NC(=O)CS[*:1]            predicted change 2.03

0 of the 595 pairs of groups are separated by more than twice 

This is the part of the table that holds whatever the ranking says, and on this set it gives a
blunt answer. The largest predicted change among all 595 pairs of groups is 2.09 log units, and
twice the conformal half width is 2.60. So the model separates **no pair of groups at all** on
this core: every one of the 35 groups has partners it cannot be told apart from.

That is not a statement about the chemistry. It is the model's own resolution: with a q2 of 0.53
on 66 molecules, a 90 % interval is 1.30 log units wide on each side, and the exchanges on this
core are smaller than that. A table of this kind becomes useful when the model's error is small
next to the changes it predicts, which means more molecules or a narrower question.

Read in that light, the table still answers the chemist's question, only with a wide tolerance:
*as far as this model can see, these groups are interchangeable.* That is honest, and it is
worth more than a confident ranking that memory produced.

## What to take from this notebook

- A bioisostere table can be read off a fitted model without refitting it once per group,
  because the prediction of the overlap kernel is a sum over the atoms.
- A distance between two groups, even one that compares them in their poses, ranks the measured
  activity difference at about zero. Whether a difference matters depends on where it sits.
- The co-field's predicted change ranks it well, but a kernel ridge model on fingerprints ranks
  it as well: +0.52 against +0.56 on this set, and ahead of the field model on the other two.
  All three intervals contain zero. What the co-field gains over a distance is the fitting.
- Refit without both members of a pair before judging a predicted change. With the whole model
  the same numbers are +0.63 to +0.71 instead of +0.28 to +0.56, and that difference is memory.
- The number a chemist should read is the size of the predicted change against twice the
  conformal half width. On this set no pair of groups clears it, which says the model is too
  coarse for this core, not that the groups are equal.

From the command line:

```
cmf build examples/sutherland/GPB.sdf --activity pKi --save gpb.cmf
```

```
cmf isosteres gpb.cmf --molecules examples/sutherland/GPB.sdf --activity pKi --csv table.csv
```

The command finds the position carrying the most different groups by itself, and marks the pairs
the model cannot separate.

Where to read further in the manual: *Which groups are interchangeable: a bioisostere table* has
the decomposition, the table and this measurement; *How certain is one prediction? Conformal
intervals* has the half width the table is read against.